# مطابقة الفيديو مع السكريبت
**قبل البدء:** من القائمة اختر *Runtime ← Change runtime type ← T4 GPU*، ثم شغّل الخلايا بالترتيب (زر ▶ لكل خلية).


In [ ]:
!pip -q install faster-whisper sentence-transformers gradio pillow yt-dlp

In [ ]:
%%writefile core.py
"""Pipeline: ASR (Whisper) + text embeddings + CLIP frames -> match script -> cut & concat."""
import os, re, json, shutil, subprocess, traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

DATA = Path(os.getenv("DATA_DIR", "/data"))
DEVICE = os.getenv("DEVICE", "cpu")
WHISPER_MODEL = os.getenv("WHISPER_MODEL", "large-v3" if DEVICE == "cuda" else "small")
COMPUTE = os.getenv("COMPUTE_TYPE", "float16" if DEVICE == "cuda" else "int8")
BEAM = int(os.getenv("BEAM_SIZE", "5" if DEVICE == "cuda" else "1"))
VCODEC = os.getenv("VIDEO_CODEC", "libx264")      # h264_nvenc على GPU
RENDER_WORKERS = int(os.getenv("RENDER_WORKERS", "3"))
STEP = 2.0                                        # دقة الشبكة الزمنية بالثواني
MIN_CLIP, MAX_CLIP = 4.0, 30.0

STATE = {}
HOOK = None


def set_state(job_id, **kw):
    STATE.setdefault(job_id, {}).update({k: str(v) for k, v in kw.items()})
    if HOOK:
        HOOK(STATE[job_id])


def get_state(job_id):
    return STATE.get(job_id, {})


def load_plan(job_id):
    p = DATA / "jobs" / job_id / "plan.json"
    return json.loads(p.read_text("utf-8")) if p.exists() else []


def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(f"{cmd[0]} failed: {p.stderr[-800:]}")
    return p.stdout


_models = {}
def cached(name, factory):
    if name not in _models:
        _models[name] = factory()
    return _models[name]


def probe(path):
    j = json.loads(run(["ffprobe", "-v", "error", "-show_entries",
                        "format=duration:stream=codec_type", "-of", "json", str(path)]))
    streams = [s.get("codec_type") for s in j.get("streams", [])]
    if "video" not in streams:
        raise RuntimeError("الملف لا يحتوي على مسار فيديو")
    return float(j["format"]["duration"]), "audio" in streams


# ---------------------------------------------------------------- script
def split_script(text, target):
    text = re.sub(r"[ \t]+", " ", text).strip()
    sents = [s.strip() for s in re.split(r"(?<=[.!?؟…])\s+|\n+", text) if s.strip()]
    if not sents:
        raise RuntimeError("السكريبت فارغ")
    max_segments = max(1, int(target // MIN_CLIP))
    min_words = 15
    while True:
        groups, cur, n = [], [], 0
        for s in sents:
            cur.append(s); n += len(s.split())
            if n >= min_words:
                groups.append(" ".join(cur)); cur, n = [], 0
        if cur:
            if groups and n < min_words / 2:
                groups[-1] += " " + " ".join(cur)
            else:
                groups.append(" ".join(cur))
        if len(groups) <= max_segments:
            return groups
        min_words = int(min_words * 1.5) + 1


def allocate(groups, target, avail):
    w = np.array([max(1, len(g.split())) for g in groups], float)
    d = w / w.sum() * target
    for _ in range(20):
        d = np.clip(d * target / d.sum(), MIN_CLIP, MAX_CLIP)
    return np.minimum(d, avail)


# ---------------------------------------------------------------- models
def transcribe(audio, language, total, on_progress):
    from faster_whisper import WhisperModel
    m = cached("whisper", lambda: WhisperModel(WHISPER_MODEL, device=DEVICE, compute_type=COMPUTE))
    segs, _ = m.transcribe(str(audio), language=language or None, vad_filter=True,
                           beam_size=BEAM, condition_on_previous_text=False)
    out = []
    for s in segs:
        t = s.text.strip()
        if t:
            out.append((float(s.start), float(s.end), t))
        on_progress(min(1.0, s.end / max(total, 1)))
    return out


def text_scores(queries, segs, G):
    from sentence_transformers import SentenceTransformer
    m = cached("txt", lambda: SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=DEVICE))
    Q = m.encode(queries, normalize_embeddings=True, convert_to_numpy=True, batch_size=64)
    E = m.encode([s[2] for s in segs], normalize_embeddings=True, convert_to_numpy=True, batch_size=64)
    sims = Q @ E.T
    grid = np.full((len(queries), G), sims.min(), dtype=np.float32)
    for j, (s, e, _) in enumerate(segs):
        g0, g1 = min(G - 1, int(s // STEP)), min(G - 1, int(e // STEP))
        grid[:, g0:g1 + 1] = np.maximum(grid[:, g0:g1 + 1], sims[:, j:j + 1])
    return grid


def visual_scores(video, tmp, queries, G):
    from PIL import Image
    from sentence_transformers import SentenceTransformer
    fdir = tmp / "frames"; fdir.mkdir(parents=True, exist_ok=True)
    run(["ffmpeg", "-v", "error", "-i", str(video), "-vf", f"fps=1/{STEP},scale=-2:224",
         "-q:v", "4", str(fdir / "%06d.jpg")])
    files = sorted(fdir.glob("*.jpg"))
    if not files:
        return None
    img_m = cached("clip_img", lambda: SentenceTransformer("clip-ViT-B-32", device=DEVICE))
    txt_m = cached("clip_txt", lambda: SentenceTransformer(
        "sentence-transformers/clip-ViT-B-32-multilingual-v1", device=DEVICE))
    embs = []
    for i in range(0, len(files), 256):
        imgs = []
        for f in files[i:i + 256]:
            with Image.open(f) as im:
                imgs.append(im.convert("RGB"))
        embs.append(img_m.encode(imgs, batch_size=64, normalize_embeddings=True, convert_to_numpy=True))
    E = np.concatenate(embs)
    Q = txt_m.encode(queries, normalize_embeddings=True, convert_to_numpy=True)
    S = (Q @ E.T).astype(np.float32)
    if S.shape[1] >= G:
        return S[:, :G]
    pad = np.repeat(S.min(axis=1, keepdims=True), G - S.shape[1], axis=1)
    return np.concatenate([S, pad], axis=1)


def zrow(x):
    return (x - x.mean(axis=1, keepdims=True)) / (x.std(axis=1, keepdims=True) + 1e-6)


# ---------------------------------------------------------------- matching
def pick_windows(score, durs, segs, dur):
    """اختيار أفضل نافذة زمنية لكل مقطع سكريبت دون تداخل بين اللقطات."""
    k, G = score.shape
    used = np.zeros(G, dtype=np.int32)
    best = [float(score[i].max()) for i in range(k)]
    order = sorted(range(k), key=lambda i: -best[i])      # الأقوى مطابقةً يختار أولاً
    seg_starts = np.array([s[0] for s in segs]) if segs else np.array([])
    seg_ends = np.array([s[1] for s in segs]) if segs else np.array([])
    res = {}
    for i in order:
        d = float(durs[i])
        w = max(1, min(G, int(round(d / STEP))))
        c = np.concatenate([[0.0], np.cumsum(score[i])])
        win = (c[w:] - c[:-w]) / w
        u = np.concatenate([[0], np.cumsum(used)])
        free = (u[w:] - u[:-w]) == 0
        if free.any():
            win = np.where(free, win, -np.inf)
        g = int(np.argmax(win))
        start = g * STEP
        if len(seg_starts):                                   # التقاط بداية الجملة
            near = np.abs(seg_starts - start)
            if near.min() <= 3.0:
                start = float(seg_starts[int(near.argmin())])
        start = max(0.0, min(start, dur - d))
        end = start + d
        if len(seg_ends):                                     # إنهاء عند نهاية جملة (للأسفل فقط)
            ok = seg_ends[(seg_ends <= end) & (seg_ends >= start + 0.6 * d)]
            if len(ok):
                end = float(ok.max())
        res[i] = (start, end, float(win[g]) if np.isfinite(win[g]) else 0.0)
        used[max(0, int(start // STEP) - 1): int(end // STEP) + 2] += 1
    return [res[i] for i in range(k)]


# ---------------------------------------------------------------- render
def render_clip(video, start, dur, out, has_audio, fx=False):
    vf = ("scale=1280:720:force_original_aspect_ratio=decrease,"
          "pad=1280:720:(ow-iw)/2:(oh-ih)/2,setsar=1,fps=30,format=yuv420p")
    cmd = ["ffmpeg", "-v", "error", "-y", "-ss", f"{start:.3f}", "-t", f"{dur:.3f}", "-i", str(video)]
    if has_audio:
        amap = "0:a:0"
    else:
        cmd += ["-f", "lavfi", "-t", f"{dur:.3f}", "-i", "anullsrc=r=48000:cl=stereo"]
        amap = "1:a:0"
    af = "aresample=48000,aformat=channel_layouts=stereo"
    if fx and dur > 2:
        f = 0.4
        vf += f",fade=t=in:st=0:d={f},fade=t=out:st={dur - f:.3f}:d={f}"
        af += f",afade=t=in:st=0:d={f},afade=t=out:st={dur - f:.3f}:d={f}"
    cmd += ["-map", "0:v:0", "-map", amap, "-vf", vf, "-af", af]
    cmd += ["-c:v", VCODEC] + (["-preset", "veryfast", "-crf", "20"] if VCODEC == "libx264" else [])
    cmd += ["-c:a", "aac", "-b:a", "160k", "-ar", "48000", "-ac", "2", "-t", f"{dur:.3f}", str(out)]
    run(cmd)


# ---------------------------------------------------------------- job
def process_job(job_id):
    jd = DATA / "jobs" / job_id
    tmp = jd / "tmp"
    try:
        _pipeline(job_id, jd, tmp)
    except Exception as e:
        traceback.print_exc()
        set_state(job_id, status="failed", stage="فشلت المعالجة", error=str(e)[:500])
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def _pipeline(job_id, jd, tmp):
    tmp.mkdir(exist_ok=True)
    st = get_state(job_id)
    target = int(st["target"]); language = st.get("language") or None
    use_visual = st.get("use_visual") == "1"
    video = jd / "source.bin"

    set_state(job_id, status="processing", stage="فحص الفيديو", progress=2)
    dur, has_audio = probe(video)
    target = min(target, int(dur))
    G = max(1, int(dur // STEP))
    script = (jd / "script.txt").read_text("utf-8")
    groups = split_script(script, target)
    durs = allocate(groups, target, dur)

    segs = []
    if has_audio:
        set_state(job_id, stage="استخراج الصوت", progress=4)
        wav = tmp / "audio.wav"
        run(["ffmpeg", "-v", "error", "-y", "-i", str(video), "-vn", "-ac", "1", "-ar", "16000", str(wav)])
        set_state(job_id, stage="تفريغ الكلام (Whisper)", progress=5)
        segs = transcribe(wav, language, dur,
                          lambda p: set_state(job_id, progress=5 + int(p * 45)))

    set_state(job_id, stage="تحليل التطابق النصي", progress=52)
    comps = []
    if segs:
        comps.append((0.6 if use_visual else 1.0, zrow(text_scores(groups, segs, G))))
    if use_visual or not segs:
        set_state(job_id, stage="تحليل اللقطات بصرياً (CLIP)", progress=58)
        v = visual_scores(video, tmp, groups, G)
        if v is not None:
            comps.append((0.4 if segs else 1.0, zrow(v)))
    if not comps:
        raise RuntimeError("تعذر تحليل الفيديو (لا صوت ولا إطارات)")
    score = sum(w * s for w, s in comps) / sum(w for w, _ in comps)

    set_state(job_id, stage="اختيار أفضل اللقطات", progress=75)
    wins = pick_windows(score, durs, segs, dur)
    plan = [{"index": i, "script": groups[i], "start": round(s, 2), "end": round(e, 2),
             "score": round(sc, 3)} for i, (s, e, sc) in enumerate(wins)]
    (jd / "plan.json").write_text(json.dumps(plan, ensure_ascii=False, indent=1), "utf-8")

    if st.get("render") == "0":
        set_state(job_id, status="done", stage="اكتمل (قائمة اللقطات فقط)", progress=100,
                  final_duration=round(sum(p["end"] - p["start"] for p in plan), 1))
        return
    set_state(job_id, stage="قص اللقطات", progress=78)
    cdir = tmp / "clips"; cdir.mkdir()
    outs = [cdir / f"{i:04d}.mp4" for i in range(len(plan))]
    done = 0
    with ThreadPoolExecutor(RENDER_WORKERS) as ex:
        futs = [ex.submit(render_clip, video, p["start"], p["end"] - p["start"], o, has_audio, st.get("fx") == "1")
                for p, o in zip(plan, outs)]
        for f in futs:
            f.result()
            done += 1
            set_state(job_id, progress=78 + int(done / len(futs) * 18))

    set_state(job_id, stage="دمج الفيديو النهائي", progress=97)
    lst = tmp / "list.txt"
    lst.write_text("".join(f"file '{o}'\n" for o in outs), "utf-8")
    final_tmp = jd / "final.partial.mp4"
    run(["ffmpeg", "-v", "error", "-y", "-f", "concat", "-safe", "0", "-i", str(lst),
         "-c", "copy", "-movflags", "+faststart", str(final_tmp)])
    os.replace(final_tmp, jd / "final.mp4")
    total = sum(p["end"] - p["start"] for p in plan)
    set_state(job_id, status="done", stage="اكتمل", progress=100, final_duration=round(total, 1))


**اختياري:** لفيديو كبير، ضعه في Google Drive وشغّل الخلية التالية ثم اكتب مساره في الواجهة.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**التشغيل:** بعد تشغيل الخلية التالية افتح الرابط `gradio.live` الذي سيظهر.

In [ ]:
import os, uuid
import torch
os.environ["DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["DATA_DIR"] = "/content/data"
import core, gradio as gr

print("الجهاز:", os.environ["DEVICE"], "| النموذج:", core.WHISPER_MODEL)
def fmt(s):
    s = int(s); h, m, sec = s // 3600, s % 3600 // 60, s % 60
    return f"{h}:{m:02d}:{sec:02d}" if h else f"{m}:{sec:02d}"

def fetch_url(url):
    import yt_dlp
    out = "/content/downloads"
    os.makedirs(out, exist_ok=True)
    opts = {"format": "bv*[height<=720]+ba/b[height<=720]/b", "merge_output_format": "mp4",
            "outtmpl": out + "/%(id)s.%(ext)s", "noplaylist": True, "quiet": True}
    with yt_dlp.YoutubeDL(opts) as y:
        info = y.extract_info(url, download=True)
        path = y.prepare_filename(info)
    mp4 = os.path.splitext(path)[0] + ".mp4"
    return mp4 if os.path.exists(mp4) else path

def run(video, url, vpath, script, target, lang, use_visual, render, fx, progress=gr.Progress()):
    url = (url or "").strip()
    if url:
        progress(0.01, desc="تنزيل الفيديو من الرابط")
        try:
            vpath = fetch_url(url)
        except Exception as e:
            raise gr.Error("تعذر التنزيل من الرابط (YouTube يحجب Colab أحياناً). نزّل الفيديو بنفسك وضعه في Drive. " + str(e)[:200])
    src = (vpath or "").strip() or video
    if not src or not os.path.exists(src):
        raise gr.Error("ارفع الفيديو أو اكتب مساره الصحيح")
    if len((script or "").strip()) < 20:
        raise gr.Error("السكريبت قصير جداً")
    job = uuid.uuid4().hex
    jd = core.DATA / "jobs" / job
    jd.mkdir(parents=True)
    os.symlink(os.path.abspath(src), jd / "source.bin")
    (jd / "script.txt").write_text(script, "utf-8")
    core.set_state(job, status="queued", target=int(target), language=lang or "",
                   use_visual=int(use_visual), render=int(render), fx=int(fx))
    core.HOOK = lambda s: progress(float(s.get("progress", 0)) / 100, desc=s.get("stage", ""))
    core.process_job(job)
    st = core.get_state(job)
    if st.get("status") != "done":
        raise gr.Error(st.get("error", "فشلت المعالجة"))
    plan = core.load_plan(job)
    lines = [f"{p['index']+1}) من {fmt(p['start'])} إلى {fmt(p['end'])}  ←  {p['script'][:110]}" for p in plan]
    rows = [[p["index"] + 1, fmt(p["start"]), fmt(p["end"]), p["script"][:140]] for p in plan]
    final = str(jd / "final.mp4") if render else None
    return "\n".join(lines), rows, final, final

with gr.Blocks(title="مطابقة الفيديو مع السكريبت") as demo:
    gr.Markdown("## يسمع الفيديو ويشوفه، ويحدد لك اللقطات المناسبة للسكريبت (بالدقيقة والثانية)")
    with gr.Row():
        with gr.Column():
            video = gr.File(label="ارفع الفيديو (للملفات الكبيرة استعمل المسار من Drive)", file_types=["video"], type="filepath")
            url = gr.Textbox(label="رابط الفيديو (YouTube أو رابط مباشر) - الأسهل", placeholder="https://www.youtube.com/watch?v=...")
            vpath = gr.Textbox(label="أو مسار الفيديو (مثال: /content/drive/MyDrive/video.mp4)")
            script = gr.Textbox(label="السكريبت (بأي لغة)", lines=10)
            target = gr.Slider(30, 1800, value=300, step=10, label="المدة النهائية (ثانية)")
            lang = gr.Dropdown([("اكتشاف تلقائي", ""), ("العربية", "ar"), ("English", "en"), ("Français", "fr")], value="", label="لغة الكلام في الفيديو")
            vis = gr.Checkbox(True, label="تحليل الصورة أيضاً (CLIP) - أدق")
            render = gr.Checkbox(True, label="أنتج الفيديو النهائي (إذا ألغيته: غير قائمة اللقطات، وأسرع)")
            fx = gr.Checkbox(True, label="مؤثرات انتقال ناعمة (Fade) بين اللقطات")
            btn = gr.Button("ابدأ", variant="primary")
        with gr.Column():
            times = gr.Textbox(label="اللقطات المختارة (انسخها)", lines=12, show_copy_button=True)
            plan = gr.Dataframe(headers=["#", "من", "إلى", "جزء السكريبت"], wrap=True)
            outv = gr.Video(label="الفيديو النهائي")
            outf = gr.File(label="تحميل")
    btn.click(run, [video, url, vpath, script, target, lang, vis, render, fx], [times, plan, outv, outf])

demo.queue().launch(share=True, debug=True)
